In [1]:
# %%
# --- Load FROZEN code embeddings + verify hash (top of 05 and 06) ------------
# Guarantees every run consumes the exact same embeddings (hash 42b5429...),
# so results no longer drift when 04 is touched. NO API calls.
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
PROCESSED = ROOT / "data" / "processed"

def _content_hash(lk):
    h = hashlib.sha256()
    for k in sorted(lk.keys()):
        h.update(str(k).encode())
        h.update(np.asarray(lk[k], dtype=np.float32).tobytes())
    return h.hexdigest()[:16]

def load_frozen_lookup():
    lk = pd.read_pickle(PROCESSED / "code_emb_lookup.FROZEN.pkl")
    want = (PROCESSED / "code_emb_lookup.FROZEN.sha256").read_text().strip()
    got = _content_hash(lk)
    assert got == want, f"FROZEN embeddings changed! {got} != {want} — do NOT proceed"
    print(f"frozen embeddings verified ({got}); {len(lk)} codes")
    return lk

code_lookup = load_frozen_lookup()   # ← 05 uses this instead of code_emb_lookup.pkl

frozen embeddings verified (42b542915055e17c); 19 codes


In [2]:
# %%
# =============================================================================
# Paper C · 06_signal_decomp  (RQ2 — the paper's core contribution)
# Decompose intra-risk (firm's own lawsuit signal) vs contagion (relational
# propagation), and test whether widening relations 0–11 → 0–19 adds contagion.
#
# Conditions (identical features per node; only PROPAGATION differs):
#   S0 intra_only  : company lawsuit-aggregate features, NO graph (MLP on x)
#   S1 +contagion_0_11 : hetero-GNN propagating over relations 0–11 (paper's set)
#   S2 +contagion_0_19 : hetero-GNN propagating over ALL relations 0–19 (ours)
#
# Read:
#   S1 − S0  = contagion uplift over intra alone (is relational risk real?)
#   S2 − S1  = extra contagion from financial relations 12–19 (loan/deal/rent/
#              creditor) the original paper did not use
#
# 5 seeds, mask A primary. Same GNN arch/HP as 03/05.
# Inputs : smesd_hetero.pt (person-fixed) ; Outputs: artifacts/tables/06_decomp.csv
# =============================================================================
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "artifacts" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


# %%
# --- 1. Load graph -----------------------------------------------------------
data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
assert data["person"].num_nodes == 2405, "run person-fixed 01 first"
comp = data["company"]
N_COMP = comp.num_nodes

# relation groups (from SMEsD.md): 0–11 = paper's set, 12–19 = extra financial
def rel_num(et):
    # et like ('company','rel_6','company'); pull the integer after 'rel_'
    return int(et[1].split("_")[1])

ALL_EDGES = list(data.edge_types)
EDGES_0_11 = [et for et in ALL_EDGES if rel_num(et) <= 11]
EDGES_0_19 = ALL_EDGES
print(f"edge types: total={len(ALL_EDGES)}  0–11={len(EDGES_0_11)}  "
      f"0–19={len(EDGES_0_19)}")


# %%
# --- 2. Company features: base 23-d (same log+scale as 03/05) ----------------
# For a clean intra/contagion split we use the SAME node features everywhere;
# only propagation changes. (Base 23-d = 3 basic + 20 lawsuit-aggregate.)
from sklearn.preprocessing import StandardScaler

Xc = comp.x.numpy().astype(np.float64)
Xc[:, :2] = np.log1p(np.clip(Xc[:, :2], 0, None))


# %%
# --- 3. Models: MLP (no graph) for S0; HeteroGNN for S1/S2 -------------------
from torch_geometric.nn import HeteroConv, SAGEConv


class MLP(nn.Module):
    """S0 intra-only: node classifier with NO message passing."""
    def __init__(self, in_dim, hidden=64, out=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out))

    def forward(self, x):
        return self.net(x)


class HeteroGNN(nn.Module):
    """S1/S2: same arch as 03/05, but edge set is configurable."""
    def __init__(self, in_dims, edge_types, hidden=64, out=2, layers=2, dropout=0.3):
        super().__init__()
        self.drop = dropout
        self.edge_types = edge_types
        self.lin_in = nn.ModuleDict(
            {nt: nn.Linear(d, hidden) for nt, d in in_dims.items()})
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((hidden, hidden), hidden)
                        for et in edge_types}, aggr="sum")
            for _ in range(layers)])
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden, out))

    def forward(self, x_dict, ei_dict):
        h = {nt: F.relu(self.lin_in[nt](x)) for nt, x in x_dict.items()}
        for conv in self.convs:
            h = conv(h, ei_dict)
            h = {nt: F.dropout(F.relu(v), p=self.drop, training=self.training)
                 for nt, v in h.items()}
        return self.head(h["company"])


# %%
# --- 4. Metrics --------------------------------------------------------------
from sklearn.metrics import roc_auc_score, f1_score, roc_curve

def ks_stat(y, s):
    fpr, tpr, _ = roc_curve(y, s); return float(np.max(tpr - fpr))

def metric_row(y, s, thr=0.5):
    return {"AUC": roc_auc_score(y, s),
            "F1": f1_score(y, (s >= thr).astype(int), zero_division=0),
            "KS": ks_stat(y, s)}


# %%
# --- 5. Runners --------------------------------------------------------------
def _scale(train_mask):
    Xs = StandardScaler().fit(Xc[train_mask.numpy()]).transform(Xc).astype(np.float32)
    return torch.tensor(Xs)

def _cls_weight(y, tr):
    cls = torch.bincount(y[tr], minlength=2).float()
    return (cls.sum() / (2 * cls.clamp(min=1))).to(DEVICE)

def run_S0(seed, epochs=300, lr=5e-3, wd=5e-4, patience=40):
    """intra-only MLP (no graph)."""
    torch.manual_seed(seed); np.random.seed(seed)
    tr, va, te = comp.train_mask, comp.val_mask, comp.test_mask
    y = comp.y
    X = _scale(tr)
    model = MLP(X.size(1)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    w = _cls_weight(y, tr)
    best, state, wait = -1, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(model(X)[tr], y[tr], weight=w)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(model(X), 1)[:, 1].numpy()
        vauc = roc_auc_score(y[va].numpy(), p[va.numpy()])
        if vauc > best:
            best, state, wait = vauc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(state); model.eval()
    with torch.no_grad():
        p = F.softmax(model(X), 1)[:, 1].numpy()
    return metric_row(y[te.numpy()].numpy(), p[te.numpy()])

def run_GNN(edge_types, seed, epochs=200, lr=5e-3, wd=5e-4, patience=30):
    """contagion GNN over a given edge set."""
    torch.manual_seed(seed); np.random.seed(seed)
    tr, va, te = comp.train_mask, comp.val_mask, comp.test_mask
    y = comp.y
    x_dict = {"company": _scale(tr), "person": data["person"].x}
    ei_dict = {et: data[et].edge_index for et in edge_types}
    in_dims = {"company": x_dict["company"].size(1),
               "person": data["person"].x.size(1)}
    model = HeteroGNN(in_dims, edge_types).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    w = _cls_weight(y, tr)
    best, state, wait = -1, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(model(x_dict, ei_dict)[tr], y[tr], weight=w)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(model(x_dict, ei_dict), 1)[:, 1].numpy()
        vauc = roc_auc_score(y[va].numpy(), p[va.numpy()])
        if vauc > best:
            best, state, wait = vauc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(state); model.eval()
    with torch.no_grad():
        p = F.softmax(model(x_dict, ei_dict), 1)[:, 1].numpy()
    return metric_row(y[te.numpy()].numpy(), p[te.numpy()])


# %%
# --- 6. Run 3 conditions × 5 seeds -------------------------------------------
rows = []
for seed in SEEDS:
    rows.append({"condition": "S0_intra_only", "seed": seed, **run_S0(seed)})
    rows.append({"condition": "S1_contagion_0_11", "seed": seed,
                 **run_GNN(EDGES_0_11, seed)})
    rows.append({"condition": "S2_contagion_0_19", "seed": seed,
                 **run_GNN(EDGES_0_19, seed)})
    print(f"seed {seed} done")

raw = pd.DataFrame(rows)
raw.to_csv(TABLES / "06_decomp_raw.csv", index=False)


# %%
# --- 7. Aggregate + decomposition read-out -----------------------------------
summary = raw.groupby("condition").agg(
    AUC_mean=("AUC", "mean"), AUC_std=("AUC", "std"),
    F1_mean=("F1", "mean"), F1_std=("F1", "std"),
    KS_mean=("KS", "mean"), KS_std=("KS", "std")).round(4)
order = ["S0_intra_only", "S1_contagion_0_11", "S2_contagion_0_19"]
summary = summary.loc[order]
summary.to_csv(TABLES / "06_decomp.csv")

print("\n=== RQ2 decomposition (mask A, test, mean ± std / 5 seeds) ===")
for c in order:
    r = summary.loc[c]
    print(f"  {c:20s}  AUC {r.AUC_mean:.3f}±{r.AUC_std:.3f}  "
          f"F1 {r.F1_mean:.3f}±{r.F1_std:.3f}  KS {r.KS_mean:.3f}±{r.KS_std:.3f}")

s0, s1, s2 = (summary.loc[c, "AUC_mean"] for c in order)
print(f"\n  contagion uplift   (S1 − S0) = {s1 - s0:+.4f}  ← is relational risk real?")
print(f"  extra fin-relations(S2 − S1) = {s2 - s1:+.4f}  ← do rels 12–19 add?")


# %%
# --- 8. Significance tests (paired t, per-seed) ------------------------------
from scipy.stats import ttest_rel

def paired(a_cond, b_cond, label):
    a = raw[raw.condition == a_cond].sort_values("seed")["AUC"].values
    b = raw[raw.condition == b_cond].sort_values("seed")["AUC"].values
    t, p = ttest_rel(a, b)
    print(f"  {label}: ΔAUC={ (a-b).mean():+.4f}  t={t:.3f}  p={p:.4f}  "
          f"{'✓ sig' if p < 0.05 else 'ns'}")

print("\npaired t-tests (mask A, 5 seeds):")
paired("S1_contagion_0_11", "S0_intra_only", "S1 vs S0 (contagion adds)")
paired("S2_contagion_0_19", "S1_contagion_0_11", "S2 vs S1 (fin-rels add)")


# %%
# --- 9. Notes ----------------------------------------------------------------
print("""
06 complete — RQ2 signal decomposition (core contribution)
==========================================================
  Interpretation guide:
    • S1 − S0 large & significant → CONTAGION is the main value of alt-data
      (aligns with the paper's thesis, and with RQ3's negative result: value is
       in relations, not in per-lawsuit semantics).
    • S2 − S1 ≈ 0 → financial relations 12–19 are too sparse to add signal
      (recall: rel 13/14/15/19 have <15 edges each). A finding either way.
  Next: 07_case_study — trace concrete contagion paths (use idx2name from
  smesd_idmaps.pt) to illustrate HOW relational risk propagates.
""")
print("06_signal_decomp complete.")

edge types: total=20  0–11=12  0–19=20
seed 42 done
seed 43 done
seed 44 done
seed 45 done
seed 46 done
seed 47 done
seed 48 done
seed 49 done
seed 50 done
seed 51 done

=== RQ2 decomposition (mask A, test, mean ± std / 5 seeds) ===
  S0_intra_only         AUC 0.687±0.004  F1 0.608±0.017  KS 0.318±0.008
  S1_contagion_0_11     AUC 0.772±0.008  F1 0.677±0.033  KS 0.446±0.021
  S2_contagion_0_19     AUC 0.765±0.012  F1 0.673±0.034  KS 0.439±0.027

  contagion uplift   (S1 − S0) = +0.0852  ← is relational risk real?
  extra fin-relations(S2 − S1) = -0.0074  ← do rels 12–19 add?

paired t-tests (mask A, 5 seeds):
  S1 vs S0 (contagion adds): ΔAUC=+0.0852  t=29.239  p=0.0000  ✓ sig
  S2 vs S1 (fin-rels add): ΔAUC=-0.0073  t=-1.807  p=0.1042  ns

06 complete — RQ2 signal decomposition (core contribution)
  Interpretation guide:
    • S1 − S0 large & significant → CONTAGION is the main value of alt-data
      (aligns with the paper's thesis, and with RQ3's negative result: value is
       in 

In [3]:
# %%
# --- Figure export setup: create paper_C folder + save helper (run once) -----
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
PAPER_DIR = ROOT / "paper_C"
PAPER_DIR.mkdir(parents=True, exist_ok=True)

# Embed fonts as text (avoids font substitution at typesetting). EPS has no alpha.
matplotlib.rcParams.update({
    "ps.fonttype": 42, "pdf.fonttype": 42, "axes.unicode_minus": False,
})

def save_fig(fig, name, dpi=600):
    """Save as PDF + EPS + PNG (caption-free). Does not print the path."""
    for ext in ("pdf", "eps", "png"):
        fig.savefig(PAPER_DIR / f"{name}.{ext}", format=ext, dpi=dpi,
                    bbox_inches="tight")
    print(f"saved: {name}.pdf / .eps / .png")

print("figure export ready")

figure export ready


In [4]:
# %%
# --- [FIG] RQ2: intra vs contagion decomposition (core figure) --------------
import numpy as np, matplotlib.pyplot as plt

order = ["S0_intra_only", "S1_contagion_0_11", "S2_contagion_0_19"]
means = [summary.loc[c, "AUC_mean"] for c in order]
stds = [summary.loc[c, "AUC_std"] for c in order]

fig, ax = plt.subplots(figsize=(5.2, 4.2))
x = np.arange(len(order))
ax.bar(x, means, yerr=stds, capsize=5, width=0.6,
       color=["#9aa0a6", "#c0392b", "#e08e79"],
       edgecolor="black", linewidth=0.8)
for xi, m, s in zip(x, means, stds):
    ax.text(xi, m + s + 0.006, f"{m:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(["Intra-only\n(no graph)", "+Contagion\n(rel 0-11)",
                    "+Contagion\n(rel 0-19)"], fontsize=9)
ax.set_ylabel("Test AUC (mean ± std, 5 seeds)", fontsize=10)
ax.set_ylim(0.60, 0.82)
ax.grid(axis="y", ls=":", alpha=0.5)
for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
plt.tight_layout()
save_fig(fig, "fig_rq2_decomposition")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


saved: fig_rq2_decomposition.pdf / .eps / .png
